# Cross-Industry Accelerator — Load Lakehouse
### Auto-Load Dimension & Batch Fact Tables into Lakehouse Delta Tables

This notebook loads all **dimension tables** and **batch fact tables** into Fabric Lakehouse
as Delta tables, plus all **event/streaming tables** into the Eventhouse via KQL ingestion.

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Loads `dim_*` + `fact_*` (batch) tables → Lakehouse as Delta tables
3. Loads `fact_*` (event) + `stream_*` tables → Eventhouse KQL tables
4. Generates a load summary with row counts and status

> **Prerequisites:**
> 1. Run `01_Data_Ingestion.ipynb` to validate all CSV sources
> 2. Attach a **Lakehouse** to this notebook
> 3. For Eventhouse loading: fill in `EVENTHOUSE_CLUSTER_URI` and `EVENTHOUSE_DATABASE` in `00_Industry_Config`

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════
%run ./00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD DIMENSION TABLES → LAKEHOUSE DELTA
# ════════════════════════════════════════════════════════════════════════

import pandas as pd

results = []

print(f"Loading {len(DIM_TABLES)} dimension tables into Lakehouse ({LAKEHOUSE_SCHEMA})...\n")

for table_name in DIM_TABLES:
    csv_path = f"{CSV_BASE_PATH}/{table_name}.csv"
    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))
        row_count = df.count()
        full_table = f"{LAKEHOUSE_SCHEMA}.{table_name}"
        df.write.mode("overwrite").saveAsTable(full_table)
        results.append((table_name, "Dimension", row_count, len(df.columns), "✓"))
        print(f"  ✓ {table_name:<45} {row_count:>6} rows  {len(df.columns):>3} cols")
    except Exception as e:
        results.append((table_name, "Dimension", 0, 0, f"✗ {e}"))
        print(f"  ✗ {table_name:<45} ERROR: {e}")

print(f"\n✅ Dimension tables loaded: {sum(1 for r in results if r[4] == '✓')}/{len(DIM_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD BATCH FACT TABLES → LAKEHOUSE DELTA
# ════════════════════════════════════════════════════════════════════════

print(f"Loading {len(FACT_TABLES_BATCH)} batch fact tables into Lakehouse ({LAKEHOUSE_SCHEMA})...\n")

for table_name in FACT_TABLES_BATCH:
    csv_path = f"{CSV_BASE_PATH}/{table_name}.csv"
    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))
        row_count = df.count()
        full_table = f"{LAKEHOUSE_SCHEMA}.{table_name}"
        df.write.mode("overwrite").saveAsTable(full_table)
        results.append((table_name, "Fact (Batch)", row_count, len(df.columns), "✓"))
        print(f"  ✓ {table_name:<45} {row_count:>6} rows  {len(df.columns):>3} cols")
    except Exception as e:
        results.append((table_name, "Fact (Batch)", 0, 0, f"✗ {e}"))
        print(f"  ✗ {table_name:<45} ERROR: {e}")

print(f"\n✅ Batch fact tables loaded: {sum(1 for r in results if r[1] == 'Fact (Batch)' and r[4] == '✓')}/{len(FACT_TABLES_BATCH)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD EVENT & STREAMING TABLES → EVENTHOUSE (KQL)
# ════════════════════════════════════════════════════════════════════════
# Requires EVENTHOUSE_CLUSTER_URI and EVENTHOUSE_DATABASE to be set.
# If not configured, this cell will skip Eventhouse loading.

from notebookutils import mssparkutils

if not EVENTHOUSE_CLUSTER_URI or not EVENTHOUSE_DATABASE:
    print("⚠ Eventhouse not configured — skipping event/streaming table loading.")
    print("  Set EVENTHOUSE_CLUSTER_URI and EVENTHOUSE_DATABASE in 00_Industry_Config to enable.")
else:
    from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
    from azure.kusto.ingest import QueuedIngestClient, IngestionProperties, DataFormat

    access_token = mssparkutils.credentials.getToken(EVENTHOUSE_CLUSTER_URI)
    kcsb = KustoConnectionStringBuilder.with_aad_application_token_authentication(
        EVENTHOUSE_CLUSTER_URI, access_token
    )
    kusto_client = KustoClient(kcsb)

    # Pandas dtype → KQL type mapping
    PANDAS_TO_KQL = {
        "int64": "long", "int32": "int", "float64": "real", "float32": "real",
        "object": "string", "bool": "bool",
        "datetime64[ns]": "datetime", "datetime64[ns, UTC]": "datetime",
    }

    print(f"Loading {len(EVENTHOUSE_TABLES)} tables into Eventhouse ({EVENTHOUSE_DATABASE})...\n")

    for table_name in EVENTHOUSE_TABLES:
        csv_path = f"{CSV_BASE_PATH}/{table_name}.csv"
        category = "Streaming" if table_name.startswith("stream_") else "Fact (Event)"
        # Use table name without prefix as KQL table name
        kql_table = table_name.replace("fact_", "").replace("stream_", "")
        try:
            pdf = (spark.read
                   .option("header", True)
                   .option("inferSchema", True)
                   .csv(csv_path)
                   .toPandas())

            # Build KQL create-merge table DDL from Pandas dtypes
            columns = ", ".join(
                f"['{col}']: {PANDAS_TO_KQL.get(str(pdf[col].dtype), 'string')}"
                for col in pdf.columns
            )
            create_cmd = f".create-merge table {kql_table} ({columns})"
            kusto_client.execute_mgmt(EVENTHOUSE_DATABASE, create_cmd)

            # Ingest data
            ingest_kcsb = KustoConnectionStringBuilder.with_aad_application_token_authentication(
                EVENTHOUSE_CLUSTER_URI, access_token
            )
            ingest_client = QueuedIngestClient(ingest_kcsb)
            ingestion_props = IngestionProperties(
                database=EVENTHOUSE_DATABASE,
                table=kql_table,
                data_format=DataFormat.CSV,
            )
            ingest_client.ingest_from_dataframe(pdf, ingestion_properties=ingestion_props)

            results.append((table_name, category, len(pdf), len(pdf.columns), "✓"))
            print(f"  ✓ {table_name:<35} → {kql_table:<30} {len(pdf):>6} rows  {len(pdf.columns):>3} cols")

        except Exception as e:
            results.append((table_name, category, 0, 0, f"✗ {e}"))
            print(f"  ✗ {table_name:<35} → {kql_table:<30} ERROR: {e}")

    eh_ok = sum(1 for r in results if r[1] in ['Fact (Event)', 'Streaming'] and r[4] == '✓')
    print(f"\n✅ Eventhouse tables loaded: {eh_ok}/{len(EVENTHOUSE_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD SUMMARY REPORT
# ════════════════════════════════════════════════════════════════════════

summary_df = pd.DataFrame(results, columns=["Table", "Category", "Rows", "Columns", "Status"])

print(f"\n{'═'*75}")
print(f"LAKEHOUSE LOAD SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*75}")
print(summary_df.to_string(index=False))
print(f"\n{'─'*75}")
print(f"Total rows loaded:    {summary_df[summary_df['Status']=='✓']['Rows'].sum():,}")
print(f"Successful tables:    {(summary_df['Status']=='✓').sum()}")
print(f"Failed tables:        {(summary_df['Status']!='✓').sum()}")
print(f"\n✅ Lakehouse + Eventhouse loading complete.")
print(f"   Next: Run 03_Load_Warehouse.ipynb to load the Fabric Data Warehouse.")